# Versionnage et traçage des données

Dans ce TP, vous versionnez des données avec [DVC](https://dvc.org/) et gardez la trace des traitements qui les produisent. DVC sépare le versionnage des données, dont git suit les métadonnées (des fichiers `.dvc`), de leur stockage, ici celui de [DagsHub](https://dagshub.com/), qui héberge aussi votre dépôt git.

Les premières étapes reprennent les commandes de la frame « Flux de travail de bout en bout » du cours ; les suivantes comparent deux versions des données et montrent ce que DVC réexécute quand une dépendance change. Une ligne qui commence par `!` est une commande du shell, où une variable Python s'insère entre accolades (`{proprietaire}`). Les étapes s'enchaînent : chacune part du dépôt de la précédente.

## Configuration & installations — À exécuter

Cette cellule installe DVC et son extension S3, puis vous place dans le dossier du projet, `~/dvc-project`. Réexécutez-la si l'environnement redémarre.

In [ ]:
!apt-get -qq install tree
!pip install -q --no-warn-script-location dvc dvc-s3
!dvc config --global core.analytics false
!mkdir -p ~/dvc-project
%cd ~/dvc-project

## 1. Créer le dépôt git

Créez un compte sur [DagsHub](https://dagshub.com/), puis un dépôt git vierge. Une fois le dépôt créé, cliquez sur « *Get started with Data* » et récupérez votre jeton secret (*token*) dans la partie « *Connection credentials* ».

Ne le collez pas dans ce notebook : un notebook enregistré ou partagé le garderait. Rangez-le dans les secrets, sous le nom `DAGSHUB_TOKEN` :

- Colab : l'icône de clé (*Secrets*) dans la barre de gauche, *Ajouter un secret*, le nom et la valeur, puis activez *Accès au notebook*.
- Codespaces : sur GitHub, *Settings > Codespaces > Codespaces secrets > New secret*, le nom et la valeur, le dépôt `shuuchuu/labs-codespace` dans *Repository access*, puis arrêtez et redémarrez le Codespace : un secret n'arrive que dans un Codespace (re)démarré après lui, sous la forme d'une variable d'environnement.

Remplissez ensuite les variables `proprietaire` et `depot` avec votre nom d'utilisateur DagsHub et le nom du dépôt, puis exécutez les deux cellules suivantes. La première lit le jeton avec `lit_secret` (le secret de Colab, sinon la variable d'environnement) ou, sans secret, vous le demande sans l'afficher. La seconde crée le dépôt git du projet, relié à votre dépôt DagsHub : quand git s'authentifie, un assistant d'identification (*credential helper*) lui donne le jeton, lu dans la variable d'environnement `DAGSHUB_TOKEN`. Le jeton n'est donc écrit ni dans l'adresse du dépôt, ni dans `.git/config`.

In [ ]:
import os
from getpass import getpass

proprietaire = ""  # votre nom d'utilisateur DagsHub
depot = ""  # le nom de votre dépôt DagsHub


def lit_secret(nom):
  """La valeur du secret `nom`, ou None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab (Codespaces) : une variable d'environnement
    return os.environ.get(nom)
  try:
    return userdata.get(nom)
  except (
    userdata.SecretNotFoundError,
    userdata.NotebookAccessError,
    userdata.TimeoutException,
  ):
    return None


jeton = lit_secret("DAGSHUB_TOKEN")
if proprietaire and not jeton:
  jeton = getpass("Jeton DagsHub (la saisie ne s'affiche pas) : ")
# L'assistant d'identification de git le lira dans cette variable
os.environ["DAGSHUB_TOKEN"] = jeton or ""
print("Jeton DagsHub :", "trouvé" if jeton else "absent")

In [ ]:
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin https://{proprietaire}@dagshub.com/{proprietaire}/{depot}.git
# Quand git s'authentifie, cet assistant lui donne le jeton, lu dans la variable
# d'environnement : il n'est écrit ni dans l'adresse du dépôt ni dans .git/config
assistant = "!f() { echo password=$DAGSHUB_TOKEN; }; f"
!git config credential.helper '{assistant}'

## 2. Initialiser DVC et son stockage distant

Initialisez DVC dans le dépôt, puis déclarez le stockage de votre dépôt DagsHub, compatible S3, comme stockage distant **par défaut** de DVC, avec l'adresse de son serveur et vos identifiants (`dvc remote modify`). DagsHub donne ces commandes dans l'onglet « *Data* » du bouton vert « *Remote* » de votre dépôt : ajoutez-y l'option qui fait de ce stockage celui par défaut. Commitez ensuite la configuration de DVC (le dossier `.dvc`) et poussez-la sur DagsHub.

Affichez `.dvc/config` et `.dvc/config.local` : lequel git suit-il, et pourquoi les identifiants sont-ils dans l'autre ?

**Documentation utile**

- https://dvc.org/doc/command-reference/remote/modify
- https://dvc.org/doc/user-guide/data-management/remote-storage/amazon-s3

In [ ]:
# Votre code ici

### Conseils supplémentaires

Nommez le stockage `origin`, d'adresse `s3://dvc` ; `-d` en fait le stockage par défaut, celui de `dvc push` et `dvc pull` sans `-r`. L'adresse du serveur, `endpointurl`, est `https://dagshub.com/{proprietaire}/{depot}.s3`. Les deux identifiants, `access_key_id` et `secret_access_key`, valent tous deux votre jeton, et l'option `--local` les écrit dans `.dvc/config.local`, que le `.dvc/.gitignore` créé par `dvc init` exclut de git.

```python
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{proprietaire}/{depot}.s3
!dvc remote modify origin --local access_key_id {jeton}
!dvc remote modify origin ... secret_access_key {jeton}
!cat .dvc/config .dvc/config.local
!git add .dvc
!git commit -m "Initialise DVC"
!git push ...
```

### Solution

In [ ]:
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{proprietaire}/{depot}.s3
!dvc remote modify origin --local access_key_id {jeton}
!dvc remote modify origin --local secret_access_key {jeton}

In [ ]:
!cat .dvc/config .dvc/config.local

In [ ]:
!git add .dvc
!git commit -m "Initialise DVC"
!git push origin main

git suit `.dvc/config`, qui décrit le stockage pour toute l'équipe. `.dvc/config.local`, que `.dvc/.gitignore` exclut, garde les identifiants sur cette machine : ils ne partent jamais dans git, et chacun y met les siens (étape 9).

## 3. Récupérer une première version des données

Récupérez avec `dvc get` le fichier `wikipedia-movie-plots/wiki_movie_plots_deduped.csv` du dépôt `https://github.com/shuuchuu/datasets`, sous le nom `data.csv`, puis faites-le suivre par DVC avec `dvc add`.

Affichez ensuite `data.csv.dvc` (`!cat data.csv.dvc`) et le cache de DVC (`!tree .dvc/cache`) : que contient le fichier `.dvc`, et sous quel nom DVC range-t-il les données dans son cache ?

**Documentation utile**

- https://dvc.org/doc/command-reference/get

In [ ]:
# Votre code ici

### Conseils supplémentaires

`dvc get <dépôt> <chemin dans le dépôt> -o <fichier>` télécharge un fichier d'un dépôt git ou DVC, sans le suivre. `dvc add <fichier>` le copie dans le cache, écrit `<fichier>.dvc` et ajoute le fichier au `.gitignore`.

```python
!dvc get https://github.com/shuuchuu/datasets ... -o data.csv
!dvc add ...
!cat data.csv.dvc
!tree .dvc/cache
```

### Solution

In [ ]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc/cache

`data.csv.dvc` ne contient que le md5 du fichier, sa taille et son chemin : c'est ce petit fichier que git versionne. Le cache range les données sous leur md5 (`files/md5/<2 premiers caractères>/<le reste>`) : DVC les retrouve à partir du `.dvc`.

## 4. Enregistrer cette version

Commitez les fichiers que `dvc add` vous a dit d'ajouter à git, marquez ce commit de l'étiquette `v1`, et poussez la branche et l'étiquette sur DagsHub. Envoyez enfin les données sur le stockage distant avec `dvc push`.

Qu'envoie `git push`, et qu'envoie `dvc push` ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`dvc add` a affiché les fichiers à ajouter à git : `data.csv.dvc` et `.gitignore`. `git push origin main v1` pousse la branche et l'étiquette d'une seule commande, et `dvc push`, sans `-r`, envoie au stockage par défaut.

```python
!git add ... .gitignore
!git commit -m "Données brutes"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git add data.csv.dvc .gitignore
!git commit -m "Données brutes"
!git tag v1

In [ ]:
!git push origin main v1
!dvc push

`git push` n'envoie que les métadonnées : `data.csv.dvc` et `.gitignore`, quelques lignes. `dvc push` envoie le fichier de données lui-même (81 Mo), depuis le cache, vers le stockage.

## 5. Modifier les données

La cellule suivante modifie les données : elle trie les lignes de `data.csv` dans l'ordre inverse, puis enregistre cette nouvelle version comme la première (étapes 3 et 4), avec l'étiquette `v2`, et l'envoie sur DagsHub (git et DVC). Exécutez-la.

In [ ]:
!sort -r data.csv -o data.csv
!dvc add data.csv
!git add data.csv.dvc
!git commit -m "Données v2"
!git tag v2
!git push origin main v2
!dvc push

Comparez les deux versions : ce que git a enregistré, avec `git diff` entre les étiquettes `v1` et `v2` sur `data.csv.dvc`, puis ce qu'en dit DVC, avec `dvc diff`, et enfin le cache de DVC (`!tree .dvc/cache`). Qu'est-ce qui a changé, et qu'est-ce qui n'a pas changé ? Revenir à la première version demandera-t-il de la télécharger à nouveau ?

**Documentation utile**

- https://git-scm.com/docs/git-diff
- https://dvc.org/doc/command-reference/diff

In [ ]:
# Votre code ici

### Conseils supplémentaires

`git diff <a> <b> -- <fichier>` compare un fichier entre deux commits ou étiquettes. `dvc diff <a> <b>` compare les données que désignent les fichiers `.dvc` de deux révisions git, et liste les fichiers ajoutés, supprimés ou modifiés (`--show-hash` affiche aussi leurs md5).

```python
!git diff ... -- data.csv.dvc
!dvc diff ...
!tree .dvc/cache
```

### Solution

In [ ]:
!git diff v1 v2 -- data.csv.dvc

In [ ]:
!dvc diff v1 v2

In [ ]:
!tree .dvc/cache

Dans `data.csv.dvc`, seule la ligne du md5 change, pas celle de la taille : les mêmes lignes, dans un autre ordre. `dvc diff` le dit du point de vue des données : `data.csv` est modifié. Le cache contient les deux versions, chacune sous son md5 : revenir à la première ne demande pas de la télécharger à nouveau.

## 6. Revenir aux données originales

Vous découvrez un problème dans les nouvelles données : revenez à la première version. Restaurez avec `git restore` le `data.csv.dvc` de l'étiquette `v1`, puis, avec `dvc checkout`, le fichier de données qu'il désigne. Commitez ce retour en arrière, avec l'étiquette `v3`, et envoyez-le sur DagsHub.

Pourquoi `dvc push` n'a-t-il rien à envoyer cette fois ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`git restore --source v1 <fichier>` remet le fichier tel qu'il était à `v1`, sans changer de branche. `dvc checkout` remet alors dans le dossier de travail, depuis le cache, les données que désignent les fichiers `.dvc`.

```python
!git restore --source ... data.csv.dvc
!dvc checkout
!cat data.csv.dvc
!git add data.csv.dvc
!git commit -m "Retour aux données v1"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git restore --source v1 data.csv.dvc
!dvc checkout

In [ ]:
!cat data.csv.dvc

In [ ]:
!git add data.csv.dvc
!git commit -m "Retour aux données v1"
!git tag v3

In [ ]:
!git push origin main v3
!dvc push

Les données de `v1` sont déjà sur le stockage, sous le même md5 : DVC n'envoie que ce que le stockage n'a pas. L'historique git garde les trois versions, et `v2` reste récupérable de la même façon.

## 7. Une étape de pipeline

La cellule suivante écrit le script `upper.py`, qui recopie le fichier donné en premier argument dans celui donné en second, en majuscules. Exécutez-la.

Ajoutez avec `dvc stage add` une étape `uppercase` qui produit `data-upper.csv` à partir de `data.csv` avec ce script, en déclarant le script et les données comme dépendances. Exécutez-la avec `dvc repro`, puis enregistrez dans git le script et la pipeline, avec le fichier `dvc.lock` que `dvc repro` écrit, et envoyez sa sortie sur le stockage.

Relancez ensuite `dvc repro` : que fait-il, et pourquoi ? Que contient `dvc.lock` ?

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

texte = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(texte.upper(), encoding="utf-8")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les options de la frame « Définir une étape » du cours : `--name` (le nom de l'étape), `--deps` (une fois par dépendance), `--outs` (la sortie), puis la commande. `dvc stage add` écrit `dvc.yaml` et ajoute la sortie au `.gitignore` ; `dvc repro` exécute l'étape et écrit `dvc.lock`.

```python
!dvc stage add --name uppercase --deps ... --deps ... --outs ... python upper.py data.csv data-upper.csv
!dvc repro
!git add upper.py dvc.yaml ... .gitignore
!git commit -m "Pipeline en majuscules"
!git push origin main
!dvc push
!dvc repro
!cat dvc.lock
```

### Solution

In [ ]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

In [ ]:
!dvc repro

In [ ]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Pipeline en majuscules"
!git push origin main
!dvc push

In [ ]:
!dvc repro
!cat dvc.lock

Le second `dvc repro` saute l'étape : il compare les md5 de ses dépendances et de sa sortie à ceux de `dvc.lock`, et rien n'a changé. `dvc.lock` note la commande et ces md5 : il dit exactement à partir de quoi et comment `data-upper.csv` a été produit, c'est le traçage des données.

## 8. Modifier une dépendance

La cellule suivante réécrit le script `upper.py` : il affiche en plus le nombre de lignes qu'il traite. Exécutez-la.

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

texte = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(texte.upper(), encoding="utf-8")
print(f"{len(texte.splitlines())} lignes mises en majuscules")

Demandez à DVC, avec `dvc status`, si la pipeline est à jour : quelle étape est périmée, et pourquoi ? Relancez-la avec `dvc repro` : que réexécute-t-il, et que saute-t-il ? Vérifiez avec `dvc status`, puis regardez avec `git diff` ce qui a changé dans `dvc.lock` : la sortie `data-upper.csv` a-t-elle changé ?

Enregistrez enfin dans git le script et le nouveau `dvc.lock`, et envoyez-les sur DagsHub (git et DVC).

**Documentation utile**

- https://dvc.org/doc/command-reference/status

In [ ]:
# Votre code ici

### Conseils supplémentaires

`dvc status` compare les md5 des dépendances et des sorties de chaque étape à ceux de `dvc.lock`, et liste, pour chaque étape périmée, ce qui a changé (`changed deps`, `changed outs`). `dvc repro` exécute les étapes dont une dépendance ou un paramètre a changé (frame « Exécution » du cours), et met `dvc.lock` à jour. `git diff dvc.lock` montre ce qui a changé dans `dvc.lock` depuis le dernier commit.

```python
!dvc status
!dvc repro
!dvc status
!git diff dvc.lock
!git add ... dvc.lock
!git commit -m "upper.py compte les lignes"
!git push origin main
!dvc push
```

### Solution

In [ ]:
!dvc status

In [ ]:
!dvc repro
!dvc status

In [ ]:
!git diff dvc.lock

In [ ]:
!git add upper.py dvc.lock
!git commit -m "upper.py compte les lignes"
!git push origin main
!dvc push

`dvc status` signale l'étape `uppercase` : sa dépendance `upper.py` a changé (`modified: upper.py`), pas `data.csv`. `dvc repro` ne réexécute que cette étape et saute `data.csv.dvc`, qui n'a pas changé : DVC ne réexécute que les étapes dont une entrée a changé. Dans `dvc.lock`, seul le md5 de `upper.py` change : celui de `data-upper.csv` reste le même, car le script produit la même sortie. DVC ne pouvait pas le savoir sans l'exécuter : une dépendance modifiée suffit à réexécuter l'étape. `dvc push` n'a donc rien de nouveau à envoyer.

## 9. Sur une autre machine

Une collègue rejoint le projet. Clonez votre dépôt DagsHub dans `~/other-machine`, comme elle le ferait sur sa machine, et placez-vous dans ce dossier avec la commande magique `%cd`. Ni le jeton ni les identifiants du stockage ne sont passés par git : donnez au clone l'assistant d'identification de git (étape 1) et les identifiants du stockage (étape 2), puis récupérez les données avec `dvc pull`.

Le clone a-t-il `data.csv` et `data-upper.csv` ? Que disent `dvc status` et `dvc repro` ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`git clone -c credential.helper='{assistant}' <adresse> ~/other-machine`, avec l'adresse du dépôt de l'étape 1 : `-c` règle l'assistant dans le clone avant qu'il ne télécharge le dépôt (sans lui, git ne peut pas cloner un dépôt privé) ; les deux `dvc remote modify ... --local` de l'étape 2 ; `dvc pull` récupère toutes les données que désignent les fichiers `.dvc` et `dvc.lock` du commit courant.

```python
!git clone -c credential.helper='{assistant}' https://{proprietaire}@dagshub.com/{proprietaire}/{depot}.git ...
%cd ~/other-machine
!dvc remote modify origin --local ... {jeton}
!dvc remote modify origin --local ... {jeton}
!dvc pull
!ls
!dvc status
!dvc repro
```

### Solution

In [ ]:
!git clone -c credential.helper='{assistant}' https://{proprietaire}@dagshub.com/{proprietaire}/{depot}.git ~/other-machine

In [ ]:
%cd ~/other-machine

In [ ]:
!dvc remote modify origin --local access_key_id {jeton}
!dvc remote modify origin --local secret_access_key {jeton}

In [ ]:
!dvc pull

In [ ]:
!ls
!dvc status
!dvc repro

`dvc pull` a récupéré `data.csv`, dans sa version de `v3`, et `data-upper.csv`, sans réexécuter le script. `dvc status` dit que les données et la pipeline sont à jour, et `dvc repro` n'a rien à refaire : le clone a exactement les données et les traitements du projet.